# Dino Embeddings for Diversity Scoring

In [1]:
import torch
from torchvision import transforms
from PIL import Image

# load DINOv2
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
model.eval().cuda()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5,0.5,0.5), std=(0.5,0.5,0.5))
])

def embed(image_path):
    img = Image.open(image_path).convert("RGB")
    x = transform(img).unsqueeze(0).cuda()
    with torch.no_grad():
        feat = model(x)
    return feat.squeeze().cpu()

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to C:\Users\Salle-Cineradio/.cache\torch\hub\main.zip


C:\Users\Salle-Cineradio/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\Salle-Cineradio/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\Salle-Cineradio/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to C:\Users\Salle-Cineradio/.cache\torch\hub\checkpoints\dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:07<00:00, 11.7MB/s]


In [ ]:
embeddings = []
paths = []

for p in unlabeled_images:
    embeddings.append(embed(p))
    paths.append(p)

X = torch.stack(embeddings)  # [N, D]

In [ ]:
X = X / X.norm(dim=1, keepdim=True)

In [ ]:
import torch

def k_center_greedy(X, k):
    N = X.shape[0]

    # pick random first point
    selected = [torch.randint(0, N, (1,)).item()]

    # distance to nearest selected point
    dist = torch.cdist(X, X[selected]).min(dim=1).values

    for _ in range(k - 1):
        idx = torch.argmax(dist).item()
        selected.append(idx)

        new_dist = torch.cdist(X, X[[idx]]).squeeze(1)
        dist = torch.minimum(dist, new_dist)

    return selected

In [ ]:
k = 100  # batch size
chosen_idx = k_center_greedy(X, k)

selected_files = [paths[i] for i in chosen_idx]